# Week 11 Coding Practice: Does the Neural Network Earn Its Complexity?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/DataScience-book/blob/w11-cp-01-colab-link/module11/week11_neural_network_practice.ipynb)

**Estimated time:** 45-60 minutes
**Format:** ungraded guided practice

The reading chapter opened with a question: if we replace a familiar model with a neural network, what evidence would justify that choice? In this activity you will answer that question with a real comparison instead of a hypothetical one. You will fit a baseline, a logistic regression model, and a small neural network on the same classification task, compare them using the same evaluation workflow you already know, and look at a genuine training loss curve rather than a static illustration.

> This activity uses a well-known teaching dataset (the scikit-learn breast cancer dataset) to practice model comparison. It is an educational example for learning the workflow. It is **not** a clinical decision tool, and nothing in this notebook should be read as medical advice or a diagnostic claim about any real patient.

The guiding question throughout is the same one the chapter opened with:

> If we replace a familiar model with a neural network, what evidence would justify that choice?


## Learning objectives

By the end of this activity, you should be able to:

1. identify a classification target, its features, and what its class labels mean, including which class is treated as "positive";
2. create a stratified train/test split with a fixed random seed and explain why stratification matters here;
3. fit and evaluate a `DummyClassifier` baseline and a logistic regression model using a `StandardScaler` pipeline fit only on training data;
4. fit a small `MLPClassifier` (neural network) using the same pipeline approach;
5. compare accuracy, precision, recall, and confusion matrices across models, with the positive class stated explicitly;
6. read a neural network's training loss curve and state what it does and does not establish;
7. recommend a model using evidence, including complexity and limitations, rather than the single highest accuracy number.


## 1. Understand the prediction problem

The scikit-learn breast cancer dataset (the "Breast Cancer Wisconsin (Diagnostic)" dataset) contains 569 samples, each a set of 30 numeric measurements taken from a digitized image of a cell sample. The 30 features are the mean, standard error, and "worst" (largest) value of 10 base measurements (such as radius, texture, and smoothness), giving 10 x 3 = 30 columns.

### Variables used in this activity

| Variable | Role | Meaning |
| --- | --- | --- |
| `diagnosis` (the target) | response | 0 = malignant, 1 = benign |
| `mean radius`, `mean texture`, `mean smoothness`, ... | numeric predictors | mean value of each of 10 base cell measurements |
| `radius error`, `texture error`, ... | numeric predictors | standard error of each measurement |
| `worst radius`, `worst texture`, ... | numeric predictors | largest (worst) value of each measurement observed in the sample |

Notice the target's encoding: **0 means malignant, 1 means benign.** This is easy to misread, and it matters later in this notebook when we compute precision and recall.

The model below is not fit yet; building, fitting, and comparing the models is the point of this activity.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

plt.style.use("seaborn-v0_8-whitegrid")
RANDOM_STATE = 42

data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

print(f"Rows: {len(X)}, features: {X.shape[1]}")
print(f"Class labels: {list(data.target_names)} (encoded as 0 and 1)")

# TODO 1: Count how many samples belong to each class using np.bincount(y).
# Store the result as class_counts (an array where class_counts[0] is the
# number of malignant samples and class_counts[1] is the number of benign
# samples).
class_counts = ...

print(f"Malignant (0): {class_counts[0]}  Benign (1): {class_counts[1]}")


### Reasoning checkpoint

Before splitting or fitting anything: are the two classes in this dataset close to balanced, or is one class notably more common? Why might that matter for how you read a model's accuracy later in this notebook?


## 2. Create a stratified train/test split

Because the two classes are not evenly balanced, a plain random split could by chance put a different proportion of malignant cases in the training set than in the test set. A **stratified** split keeps the class proportions approximately the same in both the training and test sets. We also fix a `random_state` so the split is reproducible.


In [ ]:
# TODO 2: Create a stratified train/test split.
# Use train_test_split with test_size=0.2, random_state=RANDOM_STATE, and
# stratify=y. Store the four results as X_train, X_test, y_train, y_test.
X_train, X_test, y_train, y_test = ...

print(f"Training rows: {len(X_train)}, test rows: {len(X_test)}")
print(f"Training set malignant share: {(y_train == 0).mean():.3f}")
print(f"Test set malignant share: {(y_test == 0).mean():.3f}")


## 3. Fit a baseline and a logistic regression model

Before trying a neural network, we need two simpler points of comparison:

- A **baseline** (`DummyClassifier`) that ignores the features entirely and always predicts the most frequent class. Any real model should clearly beat this, or it is not adding value.
- A **familiar model** (logistic regression), fit inside a pipeline so feature scaling happens automatically and only on training data each time the pipeline is fit.


In [ ]:
# TODO 3: Fit a DummyClassifier baseline on the training data.
# Use DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE).
# Fit it on X_train, y_train, then predict on X_test and compute test
# accuracy with accuracy_score. Store the fitted model as baseline_model
# and the test accuracy as baseline_accuracy.
baseline_model = ...
baseline_accuracy = ...

print(f"Baseline test accuracy: {baseline_accuracy:.3f}")


### Checkpoint 1: What does the baseline predict, and why is it useful?

- Which single class does the baseline predict for every row in the test set?
- Is the baseline's accuracy close to, or clearly below, the proportion of that class in the test set? Why does that make sense?
- Why is "does this model clearly beat the baseline" a more meaningful bar than "is this model's accuracy a high-looking number"?


In [ ]:
# TODO 4: Build and fit a logistic regression pipeline.
# Use Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))]).
# Fit it on X_train, y_train, then predict on X_test and compute test
# accuracy. Store the fitted pipeline as logreg_model and the test
# accuracy as logreg_accuracy.
logreg_model = ...
logreg_accuracy = ...

print(f"Baseline test accuracy:            {baseline_accuracy:.3f}")
print(f"Logistic regression test accuracy: {logreg_accuracy:.3f}")


## 4. Fit a small neural network

Neural networks are typically sensitive to the scale of their input features: a feature measured in the hundreds and a feature measured in fractions of a unit can otherwise contribute very unevenly to a neuron's weighted sum. We use the same pipeline pattern as Step 3, so scaling is fit only on training data every time the pipeline is fit, exactly as in the reading chapter's evaluation workflow (Section 11.4).

The `MLPClassifier` below also uses `early_stopping=True`, which holds out part of the *training* data internally (never the test set) to decide when to stop training, instead of training for a fixed number of iterations regardless of whether it is still improving.


In [ ]:
# TODO 5: Build and fit a small MLPClassifier pipeline.
# Use Pipeline([("scaler", StandardScaler()), ("clf", MLPClassifier(
#     hidden_layer_sizes=(16, 8),
#     learning_rate_init=0.01,
#     early_stopping=True,
#     validation_fraction=0.15,
#     n_iter_no_change=30,
#     max_iter=1000,
#     random_state=RANDOM_STATE,
# ))]).
# Fit it on X_train, y_train, then predict on X_test and compute test
# accuracy. Store the fitted pipeline as mlp_model and the test accuracy
# as mlp_accuracy.
mlp_model = ...
mlp_accuracy = ...

print(f"Baseline test accuracy:            {baseline_accuracy:.3f}")
print(f"Logistic regression test accuracy: {logreg_accuracy:.3f}")
print(f"Neural network test accuracy:      {mlp_accuracy:.3f}")


### Checkpoint 2: Why should scaling be inside a pipeline?

- If you had instead called `StandardScaler().fit_transform(X)` on the *full* dataset before splitting into training and test sets, what information from the test set would leak into training?
- The pipeline calls `.fit(X_train, y_train)` once. What two things does that single call actually do, in order?
- When the pipeline later calls `.predict(X_test)`, is the scaler re-fit on the test data, or does it reuse the scaling it already learned from training data? Why does that distinction matter?


## 5. Compare accuracy, precision, recall, and confusion matrices

Remember from Step 1: this dataset encodes the target as **0 = malignant, 1 = benign**. Scikit-learn's `precision_score` and `recall_score` treat class `1` as the "positive" class by default, which here would silently mean *benign is positive*.

In a screening context, the clinically costly error is a **false negative for malignant**: a malignant case that the model predicts as benign. We therefore compute precision and recall with `pos_label=0`, treating malignant as the positive class we care about, and state that choice explicitly rather than accepting the library default.


In [ ]:
# TODO 6: For each of the three fitted models (baseline_model, logreg_model,
# mlp_model), compute test-set accuracy, precision, and recall, with
# malignant (0) as the positive class (pos_label=0). Build a DataFrame
# named comparison_table with one row per model and columns "accuracy",
# "precision_malignant", "recall_malignant". Also compute a confusion
# matrix for each model (labels=[0, 1]) and store the three matrices as
# baseline_cm, logreg_cm, and mlp_cm.
comparison_table = ...
baseline_cm = ...
logreg_cm = ...
mlp_cm = ...

display(comparison_table.round(3))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, cm, title in zip(
    axes,
    [baseline_cm, logreg_cm, mlp_cm],
    ["Baseline", "Logistic regression", "Neural network"],
):
    ConfusionMatrixDisplay(cm, display_labels=["malignant", "benign"]).plot(
        ax=ax, colorbar=False
    )
    ax.set_title(title)
plt.tight_layout()
plt.show()


### Checkpoint 3: Which classification errors matter in this example?

Using the confusion matrices above (rows are the actual class, columns are the predicted class, in `[malignant, benign]` order):

- For the logistic regression model and the neural network, how many malignant cases did each model incorrectly predict as benign (false negatives for malignant)? How many benign cases did each model incorrectly flag as malignant (false positives for malignant)?
- Which of those two error types is more costly in a screening context: missing an actual malignant case, or raising an unnecessary follow-up on a benign case? Why?
- Do the two models differ more in how many malignant cases they catch (recall), or in how many unnecessary false alarms they raise (precision)?


## 6. Examine the neural network's training loss curve

After fitting, an `MLPClassifier` stores its per-iteration training loss in `loss_curve_`. This shows how the loss the optimizer was minimizing changed as training progressed. It is the training loss only: scikit-learn does not automatically provide a matching validation-loss curve the way the reading chapter's conceptual overfitting figure (Section 11.4) shows.


In [ ]:
# TODO 7: Extract the training loss curve from the fitted MLPClassifier.
# Hint: the fitted classifier is mlp_model.named_steps["clf"], and it has
# a loss_curve_ attribute. Store it as mlp_loss_curve.
mlp_loss_curve = ...

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(mlp_loss_curve, color="#0072B2")
ax.set_xlabel("Training iteration")
ax.set_ylabel("Training loss")
ax.set_title("Neural network training loss curve")
plt.tight_layout()
plt.show()

print(f"Iterations until early stopping: {len(mlp_loss_curve)}")
print(f"First-iteration loss: {mlp_loss_curve[0]:.3f}")
print(f"Final loss: {mlp_loss_curve[-1]:.3f}")


### Checkpoint 4: What can and cannot be concluded from the training loss curve?

- Describe the shape of the curve in one sentence: does the loss fall quickly then level off, fall steadily throughout, or something else?
- This curve shows only *training* loss. Does a steadily falling training loss, by itself, tell you whether the model generalizes to data it has not seen? What would you need to look at instead to answer that question? (Hint: you already computed it, in Step 5.)
- Suppose the loss curve had stopped after only 3-4 iterations with the loss barely lower than where it started. Would that be evidence the model learned a simple, efficient solution, or a sign that something went wrong? What would you check next?


## 7. Recommend a model using evidence

### Quick check: critique an AI-generated claim

Suppose an AI assistant looked only at the accuracy numbers from Step 5 and wrote:

> "The neural network's test accuracy was lower than logistic regression's, so neural networks are generally worse than logistic regression for medical diagnosis tasks."

Using the comparison you built in this notebook, write a correction in a sentence or two. What does this notebook's comparison actually support a claim about, and what would it *not* be reasonable to conclude from one comparison on one dataset with one configuration?


### Checkpoint 5 (final): Does the neural network justify its added complexity?

Write a short (4-6 sentence) evidence-based recommendation, addressing each point below using numbers and observations from this notebook, not a general impression:

- How do the three models compare on accuracy, precision, and recall (malignant as positive)? (Step 5)
- What did the training loss curve show, and what did it leave unresolved? (Step 6)
- Beyond the metrics, what other costs does the neural network carry compared to logistic regression (training time, stability across reruns, how easy each model is to explain to someone else)?
- Based on all of the above, would you recommend the neural network, the logistic regression model, or say the evidence does not clearly favor either? Why?
- What is one limitation of this entire comparison (dataset size, a single train/test split, the fact that this is a teaching dataset, or something else)?

**Your response:**

*(write here)*
